# Model Comparison & Ensemble Composition

Analyzes how similar/different models are from each other:
- Pairwise model agreement and confusion matrices
- Identify redundant vs. complementary models
- Decision diversity metrics
- Model-to-model correlation

**Prerequisites:** Run `Data-Preparation.ipynb` first.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loaders import load_model_master
from src.config import get_paths

paths = get_paths()
sns.set_style("whitegrid")

model_master = load_model_master()
models = sorted(model_master['participantID'].unique())

print(f"✓ Loaded {len(models)} models:")
for m in models:
    print(f"    {m}")

## 1. Pairwise Agreement Matrix

Compute decision agreement between all model pairs.

In [ ]:
def compute_pairwise_agreement(df, models):
    """Compute agreement (proportion of matching decisions) between all model pairs."""
    n_models = len(models)
    agreement_matrix = np.zeros((n_models, n_models))
    
    for i, model_i in enumerate(models):
        for j, model_j in enumerate(models):
            if i == j:
                agreement_matrix[i, j] = 1.0
            else:
                df_i = df[df['participantID'] == model_i].set_index('stimID')['decision']
                df_j = df[df['participantID'] == model_j].set_index('stimID')['decision']
                
                # Find common trials
                common_trials = df_i.index.intersection(df_j.index)
                if len(common_trials) > 0:
                    agreement = (df_i[common_trials] == df_j[common_trials]).mean()
                    agreement_matrix[i, j] = agreement
                else:
                    agreement_matrix[i, j] = np.nan
    
    return pd.DataFrame(
        agreement_matrix,
        index=models,
        columns=models
    )

agreement_df = compute_pairwise_agreement(model_master, models)
print("Pairwise Model Agreement:")
display(agreement_df.round(3))

## 2. Agreement Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

sns.heatmap(
    agreement_df,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=0.4,
    vmax=1.0,
    ax=ax,
    square=True,
    cbar_kws={"label": "Agreement"}
)

ax.set_title("Pairwise Model Agreement (Decision Alignment)", fontsize=14)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(paths.outputs_dir / "model-agreement-heatmap.pdf", bbox_inches="tight")
plt.show()

print("✓ Saved: model-agreement-heatmap.pdf")

## 3. Decision Diversity Metrics

Quantify how much models disagree (ensemble diversity).

In [ ]:
def compute_diversity(df, models):
    """Compute average pairwise disagreement (diversity) for each condition."""
    conds = sorted(df['condition'].unique())
    
    rows = []
    for cond in conds:
        dfc = df[df['condition'] == cond]
        
        # Pairwise disagreements for this condition
        disagreements = []
        n_trials = 0
        
        for i, model_i in enumerate(models):
            for j, model_j in enumerate(models):
                if i >= j:
                    continue  # avoid duplicates
                
                df_i = dfc[dfc['participantID'] == model_i].set_index('stimID')['decision']
                df_j = dfc[dfc['participantID'] == model_j].set_index('stimID')['decision']
                
                common = df_i.index.intersection(df_j.index)
                if len(common) > 0:
                    disagreements.append((df_i[common] != df_j[common]).mean())
                    n_trials = len(common)
        
        if disagreements:
            rows.append({
                "condition": cond,
                "mean_disagreement": np.mean(disagreements),
                "std_disagreement": np.std(disagreements),
                "n_unique_pairs": len(disagreements),
                "n_trials": n_trials
            })
    
    return pd.DataFrame(rows)

diversity_df = compute_diversity(model_master, models)
print("Decision Diversity by Condition:")
display(diversity_df)

## 4. Model Accuracy Ranking

Rank models by overall accuracy across conditions.

In [ ]:
model_accuracy = []

for model in models:
    mdf = model_master[model_master['participantID'] == model]
    acc = (mdf['decision'] == mdf['TP']).mean()
    n_trials = len(mdf)
    
    model_accuracy.append({
        "model": model,
        "accuracy": acc,
        "n_trials": n_trials
    })

model_rank = pd.DataFrame(model_accuracy).sort_values("accuracy", ascending=False)
print("Model Ranking by Accuracy:")
display(model_rank)

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(model_rank["model"], model_rank["accuracy"], color="steelblue", alpha=0.7)
ax.set_xlabel("Accuracy", fontsize=12)
ax.set_title("Model Performance Ranking", fontsize=12)
ax.set_xlim([0.5, 1.0])
ax.grid(axis="x", alpha=0.3)

for i, (model, acc) in enumerate(zip(model_rank["model"], model_rank["accuracy"])):
    ax.text(acc + 0.01, i, f"{acc:.3f}", va="center")

plt.tight_layout()
plt.savefig(paths.outputs_dir / "model-accuracy-ranking.pdf", bbox_inches="tight")
plt.show()

print("✓ Saved: model-accuracy-ranking.pdf")

## 5. Complementarity Analysis

Which models make different types of errors? (Error pattern correlation)

In [ ]:
def compute_error_correlation(df, models):
    """Compute correlation of error patterns between models."""
    # Create error vectors (1 if wrong, 0 if correct)
    error_vectors = {}
    
    for model in models:
        mdf = df[df['participantID'] == model].set_index('stimID')
        error_vectors[model] = (mdf['decision'] != mdf['TP']).astype(int)
    
    # Compute pairwise correlations
    n_models = len(models)
    corr_matrix = np.zeros((n_models, n_models))
    
    for i, model_i in enumerate(models):
        for j, model_j in enumerate(models):
            if i == j:
                corr_matrix[i, j] = 1.0
            else:
                errors_i = error_vectors[model_i]
                errors_j = error_vectors[model_j]
                
                common = errors_i.index.intersection(errors_j.index)
                if len(common) > 1:
                    corr = np.corrcoef(
                        errors_i[common].values,
                        errors_j[common].values
                    )[0, 1]
                    corr_matrix[i, j] = corr if not np.isnan(corr) else 0
    
    return pd.DataFrame(corr_matrix, index=models, columns=models)

error_corr = compute_error_correlation(model_master, models)
print("Error Pattern Correlation (high = similar errors, low = complementary):")
display(error_corr.round(3))

# Visualize
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    error_corr,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    ax=ax,
    square=True,
    cbar_kws={"label": "Error Correlation"}
)

ax.set_title("Model Error Pattern Correlation (Complementarity)", fontsize=14)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(paths.outputs_dir / "model-error-correlation.pdf", bbox_inches="tight")
plt.show()

print("✓ Saved: model-error-correlation.pdf")

## 6. Condition-Specific Performance

How does each model perform across conditions?

In [ ]:
cond_perf = []

for model in models:
    mdf = model_master[model_master['participantID'] == model]
    
    for cond in sorted(mdf['condition'].unique()):
        cdf = mdf[mdf['condition'] == cond]
        acc = (cdf['decision'] == cdf['TP']).mean()
        
        cond_perf.append({
            "model": model,
            "condition": cond,
            "accuracy": acc
        })

cond_perf_df = pd.DataFrame(cond_perf)
cond_pivot = cond_perf_df.pivot(index="model", columns="condition", values="accuracy")
cond_pivot = cond_pivot[["50_50", "80_20", "100_0"]]

print("Model Accuracy by Condition:")
display(cond_pivot.round(3))

# Visualize
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(
    cond_pivot,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    vmin=0.5,
    vmax=1.0,
    ax=ax,
    cbar_kws={"label": "Accuracy"}
)

ax.set_title("Model Accuracy by Condition", fontsize=12)
ax.set_xlabel("Condition")
ax.set_ylabel("Model")
plt.tight_layout()
plt.savefig(paths.outputs_dir / "model-condition-performance.pdf", bbox_inches="tight")
plt.show()

print("✓ Saved: model-condition-performance.pdf")

## 7. Summary Report

Ensemble composition insights.